# Part 11 — Specialized Topics

**PyTimeSeries user guide** · *Hierarchical reconciliation, detection, intermittent demand, foundation models.*

> Uses the `pytimeseries` package on the shared commodity-price dataset.

In [1]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pytimeseries as pt
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Shared dataset: 356 months of global commodity prices (1992-2021)
raw = pd.read_csv("monthly_price.csv")
dates = pd.to_datetime(raw["YEAR"].astype(str) + "-" + raw["MONTH"], format="%Y-%b")
df = raw.drop(columns=["ID", "YEAR", "MONTH", "TIME"]); df.index = dates
df.index.name = "date"; df = df.asfreq("MS")
y = df["Food_Price_Index"]
print("pytimeseries", pt.__version__, "| series:", list(df.columns))

pytimeseries 0.5.0 | series: ['Urea_Price', 'MP_Price', 'TSP_Price', 'Maize_Price', 'Rice_Price', 'Wheat_Price', 'Food_Price_Index', 'Energy_Price_Index', 'VIX_Index']


In [2]:
# hierarchical reconciliation of independently-made forecasts
cols = ['Maize_Price','Rice_Price','Wheat_Price']
rec = pt.HierarchicalReconciler(bottom_ids=cols)
base = pd.DataFrame(np.random.default_rng(0).uniform(10, 30, (4, 3)), index=rec.node_ids)
out = rec.reconcile(base, method='ols')
print('coherent (Total == sum):', np.allclose(out.loc['Total'], out.loc[cols].sum()))

coherent (Total == sum): True


In [3]:
# change-point & anomaly detection
print('change-points:', [d.strftime('%Y-%m') for d in pt.detect_change_points(y, n_bkps=4)])
print('anomalies:', [d.strftime('%Y-%m') for d in pt.detect_anomalies(y, z_thresh=3).index][:5])

change-points: ['2007-05', '2010-09', '2014-11', '2020-09']
anomalies: ['2008-02', '2008-03', '2008-04', '2008-05', '2008-06']


In [4]:
# intermittent demand
rng = np.random.default_rng(0)
d = pd.Series(np.where(rng.random(80) < 0.3, rng.integers(1,6,80), 0).astype(float),
              index=pd.date_range('2015-01-01', periods=80, freq='MS'))
for M in (pt.Croston(), pt.TSB(), pt.ADIDA()):
    print(f'{type(M).__name__:8s} rate:', round(M.fit(d).predict(3).iloc[0], 3))

Croston  rate: 1.349
TSB      rate: 0.603
ADIDA    rate: 1.143


**Foundation models** (zero-shot) share the contract but need the `[foundation]` extra (weights / API key):

```python
pt.foundation.Chronos('amazon/chronos-t5-small').fit(y).predict_interval(24, level=[0.8, 0.95])
pt.foundation.TimeGPT(api_key='...').fit(y).predict(24)
```